In [3]:
import os
import sys

# Add project root (the folder that contains 'src') to sys.path so `from src...` works when
# running notebooks from the notebooks/ subfolder.

def find_project_root(start=os.getcwd()):
    cur = os.path.abspath(start)
    while True:
        if os.path.isdir(os.path.join(cur, "src")):
            return cur
        parent = os.path.dirname(cur)
        if parent == cur:
            raise FileNotFoundError("Could not find project root containing 'src'")
        cur = parent

proj_root = find_project_root()
if proj_root not in sys.path:
    sys.path.insert(0, proj_root)
print("Added to sys.path:", proj_root)


Added to sys.path: c:\Users\Krishnendu Manna\Documents\new steup\Traffic-clusturing\city-traffic-clustering


In [4]:
from src.signal_controllers import FixedTimeController
from src.metrics_tracker import MetricsTracker


In [6]:
import os
import pandas as pd
from src.traffic_environment import TrafficEnvironment

# Load required CSVs from project data/processed (adjust paths if your files are elsewhere)
scenarios_path = os.path.join(proj_root, "data", "processed", "traffic_scenarios_from_clusters_k4.csv")
traffic_data_path = os.path.join(proj_root, "data", "processed", "bangalore_traffic_processed.csv")

missing = []
if not os.path.exists(scenarios_path):
    missing.append(scenarios_path)
if not os.path.exists(traffic_data_path):
    missing.append(traffic_data_path)
if missing:
    raise FileNotFoundError("Required data files not found:\n  " + "\n  ".join(missing))

scenarios = pd.read_csv(scenarios_path)
traffic_data = pd.read_csv(traffic_data_path)
print(f"Loaded scenarios {scenarios.shape} from {scenarios_path}")
print(f"Loaded traffic_data {traffic_data.shape} from {traffic_data_path}")

# Instantiate environment so the following cell can call env.reset()
env = TrafficEnvironment(scenarios, traffic_data)


Loaded scenarios (4, 12) from c:\Users\Krishnendu Manna\Documents\new steup\Traffic-clusturing\city-traffic-clustering\data\processed\traffic_scenarios_from_clusters_k4.csv
Loaded traffic_data (8936, 14) from c:\Users\Krishnendu Manna\Documents\new steup\Traffic-clusturing\city-traffic-clustering\data\processed\bangalore_traffic_processed.csv


In [7]:
env.reset()
controller = FixedTimeController(None)
metrics = MetricsTracker()


In [8]:
for _ in range(1000):
    _, r, _, info = env.step(controller.get_action(env.current_cluster))
    metrics.update(info["avg_wait_time"], info["total_queue"], info["vehicles_passed"], r)


In [9]:
summary = metrics.get_summary()
summary


{'avg_wait': np.float64(11.42492558520626),
 'avg_queue': np.float64(16.948),
 'throughput': np.int64(986),
 'avg_reward': np.float64(0.24350744147937395)}

In [11]:
import pandas as pd
pd.DataFrame([summary]).to_csv("C:/Users/Krishnendu Manna/Documents/new steup/Traffic-clusturing/city-traffic-clustering/results/fixed_results.csv", index=False)
